# The Matrix Formulation Lab

Explore the closed-form normal equation of linear regression: constructing the design matrix, solving the normal equation, investigating matrix singularity and the pseudo-inverse under perfect collinearity, and exploring the $p \ge n$ case.

In [ ]:
import warnings
import numpy as np
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)

## 1. The Design Matrix

Let's construct a raw feature matrix $X_{raw}$ and target $y$, and prepend a column of $1$s to create the **design matrix** $X$.

In [ ]:
X_raw = np.array([[1.0, 2.0],
                  [2.0, 1.0],
                  [3.0, 4.0],
                  [4.0, 3.0],
                  [5.0, 5.0]])
y = np.array([8.0, 7.0, 15.0, 14.0, 20.0])
print("1. THE DESIGN MATRIX")
print(f"   X (5 samples x 2 features) =\n{X_raw}")
print(f"   y = {y}")

X = np.hstack([np.ones((len(X_raw), 1)), X_raw])
print(f"\n   prepend a column of ones -> the design matrix:\n{X}")
print(f"   shape {X.shape}: 3 unknowns to solve for (b0, b1, b2)")

## 2. The Normal Equation

Let's solve for the exact best-fit coefficients $\beta = (X^T X)^{-1} X^T y$ using standard matrix multiplication and verify it against scikit-learn's `LinearRegression`.

In [ ]:
print("2. THE NORMAL EQUATION   beta = (X^T X)^-1 X^T y")
XtX, Xty = X.T @ X, X.T @ y
print(f"   X^T X =\n{XtX}")
print(f"   X^T y = {Xty}")
beta = np.linalg.inv(XtX) @ Xty
print(f"\n   beta = {beta}")
print(f"   -> b0 = {beta[0]:.4f}, b1 = {beta[1]:.4f}, b2 = {beta[2]:.4f}")

sk = LinearRegression().fit(X_raw, y)
print(f"\n   scikit-learn: intercept = {sk.intercept_:.4f}, coef = {sk.coef_}")
print(f"   identical: {np.allclose(beta, np.r_[sk.intercept_, sk.coef_])}")

## 3. When $(X^T X)^{-1}$ Does Not Exist

Let's create perfect collinearity where the third column is exactly double the second column. Watch `np.linalg.inv` fail with a `Singular matrix` error, while scikit-learn uses SVD to resolve it safely with the minimum-norm pseudo-inverse solution.

In [ ]:
print("3. WHEN (X^T X)^-1 DOES NOT EXIST")
n = 50
rng = np.random.RandomState(0)
a = rng.normal(0, 1, n)
Xb = np.column_stack([np.ones(n), a, 2 * a])       # third column = 2 x second
yb = 3 * a + rng.normal(0, 0.1, n)
print(f"   feature 2 = 2 x feature 1, exactly")
print(f"   det(X^T X) = {np.linalg.det(Xb.T @ Xb):.3e}")
print(f"   rank(X) = {np.linalg.matrix_rank(Xb)}, but X has {Xb.shape[1]} columns")
try:
    np.linalg.inv(Xb.T @ Xb)
    print("   np.linalg.inv: succeeded")
except np.linalg.LinAlgError as e:
    print(f"   np.linalg.inv -> LinAlgError: {e}")

bp = np.linalg.pinv(Xb) @ yb
skb = LinearRegression().fit(Xb[:, 1:], yb)
print(f"\n   pseudo-inverse   = {bp}")
print(f"   scikit-learn     = [{skb.intercept_:.4f} {skb.coef_[0]:.4f} {skb.coef_[1]:.4f}]")
print(f"   no exception — sklearn solves with lstsq/SVD, not an explicit inverse")
print(f"   the true effect of 3.0 is split across the identical columns:")
print(f"     {bp[1]:.4f} + 2 x {bp[2]:.4f} = {bp[1] + 2 * bp[2]:.4f}")

## 4. More Features than Samples ($p \ge n$)

When you have more features than samples, OLS is guaranteed to achieve a perfect $R^2 = 1.0000$ by overfitting exactly. This score is completely meaningless.

In [ ]:
print("4. MORE FEATURES THAN SAMPLES")
Xw = rng.normal(0, 1, (3, 5))
yw = rng.normal(0, 1, 3)
print(f"   X shape {Xw.shape} -> {Xw.shape[1]} unknowns from {Xw.shape[0]} equations")
print(f"   rank(X) = {np.linalg.matrix_rank(Xw)}: infinitely many exact solutions")
print(f"   train R2 = {LinearRegression().fit(Xw, yw).score(Xw, yw):.4f}   <- meaningless")